In [1]:
print("Test")

Test


In [2]:
import os
import json
import random
import time
import pandas as pd
import estnltk
import tqdm
import pathlib
import logging
import re

logging.getLogger().setLevel(logging.WARNING)

from scripts.config import (
    SEED,
    ROOT,
    DATA_DIR,
    ENC2017_ROOT,
    UD_ET_EDT_ROOT,
    HOMONYMS_ROOT,
    ENC2017_DIRS,
    UD_ET_EDT_DIRS,
    HOMONYMS_DIRS,
    OUTPUT_DIR,
    PLOTS_DIR,
    HOMONYMS_PLOTS_DIR,
    MODEL_DIR,
)

# from pydantic import BaseModel
import tiktoken
import tempfile
from dotenv import load_dotenv
from openai import AzureOpenAI
from openai_cost_calculator import estimate_cost_typed
from typing import Any

load_dotenv(".env", verbose=True)
api_version = "2024-12-01-preview"
model_name = "gpt-4o"
deployment_name = "EstNLTK-gpt-4o"
endpoint = str(os.getenv("AZURE_ENDPOINT"))
api_key = str(os.getenv("OPENAI_API_KEY"))

### Testing


In [3]:
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

In [ ]:
# Estimate the cost of a request
max_tokens = 128
costs = {
    "gpt-4o": {
        "input": 2.50 / 1_000_000,  # $2.50 per 1M tokens
        "cached_input": 1.25 / 1_000_000,  # $1.25 per 1M tokens
        "output": 10.00 / 1_000_000,  # $10.00 per 1M tokens
    }
}

# 1) build messages
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.

Replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Preserve tense, number, case, agreement, capitalisation, punctuation, and word order as much as possible.
- Infer the token’s grammatical role and morphology from the sentence context.
- Generate 10 alternatives that fit the context and use only these cases: nominative, genitive, partitive, or additive/illative (both fit the same role in this task).
- If the token is a proper name, replace it with another plausible proper name that still fits the required case.
- Do not repeat the original token unless it is the only valid option.
"""

# Structured output format for Azure OpenAI chat completions.
# Structured Outputs require a top-level object, so the list is wrapped in a candidates field.
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "candidates",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "candidates": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 10,
                    "maxItems": 10,
                }
            },
            "required": ["candidates"],
            "additionalProperties": False,
        },
    },
}

# One-shot example to anchor the output format
one_shot_user = """Sentence: Ma näen <maja>."""

one_shot_assistant = json.dumps(
    {
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "häärberit",
        ],
    },
    ensure_ascii=False,
    indent=2,
)


# User prompt example for the real sentence
# user_prompt = """Sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots."""
# user_prompt = """Provide 10 alternatives for the marked token in the following sentence: KOMISJONI MÄÄRUS (EMÜ) nr 344/86, millega muudetakse tagatiste osas määrusi (EMÜ) nr 626/85 ja (EMÜ) 2077/85 17. veebruar 1986 EUROOPA ÜHENDUSTE KOMISJON, võttes arvesse Euroopa Majandusühenduse asutamislepingut, võttes arvesse nõukogu 14. märtsi 1977. aasta määrust (EMÜ) nr 516/77 puu- ja köögiviljatooteturu ühise korralduse kohta, 1 viimati muudetud määrusega (EMÜ) nr 3768/85, 2 eriti selle artikli 4 lõiget 8, võttes arvesse nõukogu 14. märtsi 1977. aasta määrust (EMÜ) nr 525/77, millega kehtestatakse ananassikonservide tootmistoetuste süsteem, 3 viimati muudetud määrusega (EMÜ) nr 1699/85, 4 eriti selle artiklit 8, ning arvestades, et : komisjoni 22. juuli 1985. aasta määruse (EMÜ) nr 2220/85 (millega sätestatakse põllumajandustoodete tagatissüsteemi ühised üksikasjalikud rakenduseeskirjad) 5 III jaotises sätestatakse tagatiste andmise vormid ; sama määruse artikli 19 lõikes 1 sätestatakse seoses ettemaksetega esitatud tagatiste vabastamine ; sellest tulenevalt tuleks tunnistada kehtetuks komisjoni 12. märtsi 1985. aasta määruse (EMÜ) nr 626/85 (töötlemata kuivatatud viinamarjade ja viigimarjade ostmise, müümise ja ladustamise kohta ladustusasutuste poolt) 6 ja komisjoni 25. juuli 1985. aasta määruse (EMÜ) nr 2077/85 (milles sätestatakse ananassikonservide tootmistoetuse üksikasjalikud rakenduseeskirjad) 7 vastavad sätted ; käesoleva määrusega ettenähtud meetmed on kooskõlas puu- ja köögiviljatooteturu korralduskomitee arvamusega, ON VASTU VÕTNUD KÄESOLEVA MÄÄRUSE.""",
# user_prompt = """Väljakutsete ja abisaanute analüüsi alusel otsustatakse jaanuaris-veebruaris, kas Elvas hakkavad sõitma ka kogemustega Tartu kiirabi arstid-velskrid."""
user_prompt = """Sentence: Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel."""

output_content = """{\"candidates\":[\"hüpoteesi\",\"mudeli\",\"käsitluse\",\"lähenemise\",\"kontseptsiooni\",\"idee\",\"doktriini\",\"teooriate\",\"kontseptsioonide\",\"vaatenurga\"]}"""


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": one_shot_user},
    {"role": "assistant", "content": one_shot_assistant},
    {"role": "user", "content": user_prompt},
]

# 2) token count (use model encoding)
enc = tiktoken.encoding_for_model(model_name)  # pick same model you'll use
prompt_text = "".join(m["content"] for m in messages)
prompt_tokens = (
    len(enc.encode(prompt_text)) + 32
)  # +32 for the tokens are added (316 was the actual prompt tokens)
output_tokens = (
    len(enc.encode(output_content)) + 1
)  # +1 for the assistant's role token in the response
print(f"Prompt tokens: {prompt_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Total tokens: {prompt_tokens + output_tokens}")

# 3) estimate cost
# Assuming all prompt tokens are input and cached input tokens, and max_tokens are output tokens
input_costs = costs[model_name]["input"] * prompt_tokens
print(f"Estimated input cost: ${input_costs:.6f}")
cached_input_costs = (
    costs[model_name]["cached_input"] * 0
)  # There has not been any cached input in experiments.
print(f"Estimated cached input cost: ${cached_input_costs:.6f}")
output_costs = costs[model_name]["output"] * output_tokens
print(f"Estimated output cost: ${output_costs:.6f}")
print(f"Total estimated cost: ${input_costs + cached_input_costs + output_costs:.6f}")

Prompt tokens: 299
Output tokens: 49
Total tokens: 348
Estimated input cost: $0.000748
Estimated cached input cost: $0.000000
Estimated output cost: $0.000490
Total estimated cost: $0.001238


In [28]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Hello!",
        }
    ],
    max_completion_tokens=512,
    top_p=0.95,
    model=deployment_name,
)

In [29]:
print(response)

ChatCompletion(id='chatcmpl-DZehCb459dk5GzgEFxCdrXJ5jYmje', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! 😊 How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}})], created=1777390286, model='gpt-4o-2024-11-20', object='chat.completion', service_tier='default', system_fingerprint='fp_af7f7349a4', usage=CompletionUsage(completion_tokens=11, prompt_tokens=9, total_tokens=20, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'jailbreak': {'filtered': False, 'de

In [30]:
print(response.to_dict())

{'id': 'chatcmpl-DZehCb459dk5GzgEFxCdrXJ5jYmje', 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': 'Hello! 😊 How can I assist you today?', 'refusal': None, 'role': 'assistant', 'annotations': []}, 'content_filter_results': {'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}}}], 'created': 1777390286, 'model': 'gpt-4o-2024-11-20', 'object': 'chat.completion', 'service_tier': 'default', 'system_fingerprint': 'fp_af7f7349a4', 'usage': {'completion_tokens': 11, 'prompt_tokens': 9, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'jailbreak': {'filtered': False, 'detected': False}}}]}


```
ChatCompletion(id='chatcmpl-DZZFQtM00dyhVqTaXZ2XqHnOdRB1r', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! 😊 How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}})], created=1777369344, model='gpt-4o-2024-11-20', object='chat.completion', service_tier='default', system_fingerprint='fp_af7f7349a4', usage=CompletionUsage(completion_tokens=11, prompt_tokens=9, total_tokens=20, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'jailbreak': {'filtered': False, 'detected': False}}}])
```


In [31]:
estimate_cost_typed(response).as_dict()

{'prompt_cost_uncached': '0.00002250',
 'prompt_cost_cached': '0.00000000',
 'completion_cost': '0.00011000',
 'total_cost': '0.00013250'}

In [40]:
rd = response.to_dict()
rd["cost"] = estimate_cost_typed(response).as_dict()
print(rd)

{'id': 'chatcmpl-DZehCb459dk5GzgEFxCdrXJ5jYmje', 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': 'Hello! 😊 How can I assist you today?', 'refusal': None, 'role': 'assistant', 'annotations': []}, 'content_filter_results': {'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}}}], 'created': 1777390286, 'model': 'gpt-4o-2024-11-20', 'object': 'chat.completion', 'service_tier': 'default', 'system_fingerprint': 'fp_af7f7349a4', 'usage': {'completion_tokens': 11, 'prompt_tokens': 9, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'jailbreak': {'filtered': False, 'detected': False}}}], 'cost': {'prompt_cost_uncached': '0.0000

In [42]:
# Extract used tokens from response
# Structure of the response is:
# usage=CompletionUsage(completion_tokens=11, prompt_tokens=9, total_tokens=20, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
print(f"Actual prompt tokens used: {response.usage.prompt_tokens}")
print(
    f"Actual cached input tokens used: {response.usage.prompt_tokens_details.cached_tokens}"
)
print(f"Actual completion tokens used: {response.usage.completion_tokens}")
print(f"Actual total tokens used: {response.usage.total_tokens}")
# Calculate actual cost based on usage
actual_input_cost = costs[model_name]["input"] * response.usage.prompt_tokens
actual_cached_input_cost = (
    costs[model_name]["cached_input"]
    * response.usage.prompt_tokens_details.cached_tokens
)
actual_output_cost = costs[model_name]["output"] * response.usage.completion_tokens
print(f"Input cost: ${actual_input_cost:.7f}")
print(f"Cached input cost: ${actual_cached_input_cost:.7f}")
print(f"Output cost: ${actual_output_cost:.7f}")
print(
    f"Total actual cost: ${actual_input_cost + actual_cached_input_cost + actual_output_cost:.7f}"
)

Actual prompt tokens used: 9
Actual cached input tokens used: 0
Actual completion tokens used: 11
Actual total tokens used: 20
Input cost: $0.0000225
Cached input cost: $0.0000000
Output cost: $0.0001100
Total actual cost: $0.0001325


In [33]:
client.close()

### LLM Batch request


In [19]:
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

In [16]:
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.

Replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Preserve tense, number, case, agreement, capitalisation, punctuation, and word order as much as possible.
- Infer the token’s grammatical role and morphology from the sentence context.
- Generate 10 alternatives that fit the context and use only these cases: nominative, genitive, partitive, or additive/illative (both fit the same role in this task).
- If the token is a proper name, replace it with another plausible proper name that still fits the required case.
- Do not repeat the original token unless it is the only valid option.
"""

# Structured output format for Azure OpenAI chat completions.
# Structured Outputs require a top-level object, so the list is wrapped in a candidates field.
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "candidates",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "candidates": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 10,
                    "maxItems": 10,
                }
            },
            "required": ["candidates"],
            "additionalProperties": False,
        },
    },
}

# One-shot example to anchor the output format
one_shot_user = """Sentence: Ma näen <maja>."""

one_shot_assistant = json.dumps(
    {
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "häärberit",
        ],
    },
    ensure_ascii=False,
    indent=2,
)

# Example real input sentence
# user_prompt = """Sentence: Samas on kõik uue kodu lähistel asuvad koolid sellised, mis võtavad <gümnaasiumi> vastu katsetega."""


# Create a user prompt template for the real sentences, using the same format as the few-shot example
def create_user_prompt(sentence, word_span):
    # Mark the target word with angle brackets
    start, end = word_span
    marked_sentence = (
        sentence[:start] + "<" + sentence[start:end] + ">" + sentence[end:]
    )
    return f"""Sentence: {marked_sentence}"""


def build_messages(user_prompt: str) -> list[dict[str, str]]:
    """Build the chat-completions message list for one rewrite request."""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": one_shot_user},
        {"role": "assistant", "content": one_shot_assistant},
        {"role": "user", "content": user_prompt},
    ]


def parse_response_content(content: str) -> dict[str, Any]:
    """Parse and validate the JSON payload returned by Azure OpenAI."""
    parsed = json.loads(content or "{}")
    if not isinstance(parsed, dict):
        raise ValueError("Expected a JSON object from the model response.")
    if "candidates" not in parsed or not isinstance(parsed["candidates"], list):
        raise ValueError("Model response did not contain a candidates list.")
    return parsed


def _atomic_write_json(file_path: pathlib.Path, data: list[dict[str, Any]]) -> None:
    """Write JSON atomically to avoid partial files on crash/interruption."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    temp_file_path: pathlib.Path | None = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=file_path.parent,
            prefix=f"{file_path.stem}.",
            suffix=".tmp",
            delete=False,
        ) as temp_file:
            json.dump(data, temp_file, ensure_ascii=False, indent=2)
            temp_file.flush()
            os.fsync(temp_file.fileno())
            temp_file_path = pathlib.Path(temp_file.name)

        # Atomic replace: after a crash you see either the old or the new file, never a partial one.
        os.replace(temp_file_path, file_path)
    finally:
        if temp_file_path is not None and temp_file_path.exists():
            temp_file_path.unlink(missing_ok=True)


def _load_results_json(file_path: pathlib.Path) -> list[dict[str, Any]]:
    """Load results JSON safely and fail closed on corruption to prevent silent data loss."""
    if not file_path.exists() or file_path.stat().st_size == 0:
        return []

    with open(file_path, "r", encoding="utf-8") as file_handle:
        try:
            loaded = json.load(file_handle)
        except json.JSONDecodeError as error:
            raise RuntimeError(
                f"Failed to parse existing output file {file_path}. "
                "Refusing to overwrite it to avoid data loss."
            ) from error

    if not isinstance(loaded, list):
        raise RuntimeError(
            f"Expected a JSON list in {file_path}, got {type(loaded).__name__}. "
            "Refusing to overwrite it to avoid data loss."
        )
    return loaded


def append_result_to_json(file_path, record, id=None, search_id="sentence_id"):
    """Append or update one record in a JSON array using crash-safe atomic writes."""
    resolved_path = pathlib.Path(file_path)
    data = _load_results_json(resolved_path)

    if id is not None and search_id is not None:
        # If ID is provided, update matching record; otherwise append.
        existing_record = next(
            (
                r
                for r in data
                if isinstance(r, dict)
                and r.get(search_id) is not None
                and str(r.get(search_id)) == str(id)
            ),
            None,
        )
        if existing_record:
            existing_record.update(record)
            # Remove stale error after a successful rewrite.
            existing_record.pop("error", None)
        else:
            data.append(record)
    else:
        data.append(record)

    _atomic_write_json(resolved_path, data)


def write_total_cost(file_path, total_cost):
    """Write the total cost to a separate txt file."""
    resolved_path = pathlib.Path(file_path)
    with open(resolved_path, "w", encoding="utf-8") as f:
        f.write(str(total_cost))


def rewrite_sentence_with_azure_openai(sentence_id, sentence, word_span, metadata):
    """Rewrite one sentence by replacing the marked target word via Azure OpenAI."""
    user_prompt = create_user_prompt(sentence=sentence, word_span=word_span)
    messages = build_messages(user_prompt)

    # Request a structured JSON response from Azure OpenAI chat completions.
    response = client.chat.completions.create(
        model=deployment_name,
        messages=messages,
        response_format=response_format,
        max_completion_tokens=512,
        top_p=0.95,
    )
    # Estimate cost for this response (best-effort). The estimator expects the full response object.
    try:
        cost_result = estimate_cost_typed(response)
        cost_info = cost_result.as_dict()  # Convert to dict for JSON serialization
    except Exception as _err:
        # Don't fail the rewrite on estimator errors; keep a surfacable error message.
        cost_info = {"error": str(_err)}
    message = response.choices[0].message
    refusal = getattr(message, "refusal", None)
    if refusal:
        raise RuntimeError(f"Model refused the request: {refusal}")

    # Parse the JSON response and enrich it with metadata for later analysis.
    parsed = parse_response_content(message.content or "{}")
    response_dict = response.to_dict()

    # Keep useful metadata for later analysis/debugging.
    parsed["sentence_id"] = str(sentence_id)
    parsed["source_sentence"] = sentence
    parsed["target_word"] = metadata.get("target_word", "")
    if isinstance(word_span, list):
        parsed["word_span"] = word_span
    else:
        parsed["word_span"] = word_span.astype(
            int
        ).tolist()  # Convert numpy array to list for JSON serialization
    parsed["label"] = metadata.get("label", [])
    # Attach cost estimation metadata (not part of the structured schema)
    response_dict["cost"] = cost_info

    return parsed, response_dict


# Backwards-compatible alias for any existing callers.
rewrite_sentence_with_genai = rewrite_sentence_with_azure_openai


def get_latest_processed_id(output_file):
    """Get the highest sentence ID that has already been processed and saved in the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        with open(output_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                return max(int(record["id"]) for record in data if "id" in record)
    return -1  # No valid records found, start from the beginning


def get_unprocessed_dataset(dataset, output_file):
    """Filter the dataset to include only records that have errors or have not been processed yet, based on the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        unprocessed_records = []
        processed_records = []
        # First, get the list of sentences that have errors in the output file
        with open(output_file, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
            for record in processed_data:
                if "error" in record:
                    unprocessed_records.append(record["source_sentence"])
                else:
                    processed_records.append(record["source_sentence"])
        # Now filter the original dataset to include only records that are either unprocessed or have errors
        filtered_dataset = dataset[
            dataset["sentence"].isin(unprocessed_records)
            | ~dataset["sentence"].isin(processed_records)
        ]
        return filtered_dataset
    else:
        return dataset  # If no output file exists, return the entire dataset as unprocessed


def is_transient_error(exc: Exception) -> bool:
    """Heuristically detect retryable errors such as rate limit and network hiccups."""
    message = str(exc).lower()
    transient_markers = [
        "429",
        "rate",
        "quota",
        "too many requests",
        "timeout",
        "timed out",
        "connection",
        "temporar",
        "unavailable",
        "503",
        "502",
        "504",
        "internal",
    ]
    return any(marker in message for marker in transient_markers)


def rewrite_with_retry(sentence_id, sentence, word_span, metadata, config):
    """Call rewrite function with exponential backoff on transient errors."""
    for attempt in range(1, config["max_attempts"] + 1):
        try:
            return rewrite_sentence_with_azure_openai(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                metadata=metadata,
            )
        except Exception as exc:
            # should_retry = is_transient_error(exc) and attempt < config["max_attempts"]
            should_retry = (
                attempt < config["max_attempts"]
            )  # Retry on all exceptions for robustness, but still limit the number of attempts
            if not should_retry:
                raise

            backoff = min(
                config["max_backoff_seconds"],
                config["initial_backoff_seconds"] * (2 ** (attempt - 1)),
            )
            sleep_s = backoff + random.uniform(0.0, config["jitter_seconds"])
            print(
                f"Retry {attempt}/{config['max_attempts'] - 1} for sentence_id={sentence_id} "
                f"after transient error: {exc}. Sleeping {sleep_s:.2f}s..."
            )
            time.sleep(sleep_s)


In [20]:
# Load the homonyms dataset to get the sentences we want to rewrite with Gen AI
homonyms_df = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_updated_sentences.parquet"
)
display(homonyms_df.head())
display(homonyms_df.shape)

,num,inflection_type,sentence,word,word_span,label,source
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


(7886, 7)

In [ ]:
output_file = OUTPUT_DIR / "gpt-4o-homonyms_LLM_MLM.json"  # Output file for results
# output_file_log = (
#     OUTPUT_DIR / "gpt-4o-homonyms_LLM_MLM_log.json"
# )  # More detailed log file for debugging and analysis
total_cost_txt = (
    OUTPUT_DIR / "gpt-4o-homonyms_LLM_MLM_total_cost.txt"
)  # File to store the total cost information
sample_df = homonyms_df.iloc[:].copy()  # Use N for testing
sample_df["index"] = sample_df.index.astype(
    int
)  # Ensure index is an integer for ID purposes

# gpt-4o has following rate limits:
# - 180 000 tokens per minute (TPM)
# - 1080 requests per minute (RPM)
config = {
    # Pacing configuration to avoid hitting rate limits
    "base_delay_seconds": round(60 / 1080, 2),  # RPM is 1080 for gpt-4o
    "jitter_seconds": 0.1,  # Add a small random jitter to avoid thundering herd issues
    # Retry configuration for transient failures
    "max_attempts": 6,
    "initial_backoff_seconds": 2.0,
    "max_backoff_seconds": 60.0,
}

# Start from the next unprocessed sentence based on the output file contents
unprocessed_df = get_unprocessed_dataset(sample_df, output_file)
if len(unprocessed_df) == 0:
    print("No unprocessed sentences found. All done!")
else:
    print(f"Total unprocessed sentences to process: {len(unprocessed_df)}")
    print("Next sentence to process:")
    display(unprocessed_df.head(1))

# Find processed records that are in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["sentence_id"])
            for record in processed_data
            if "sentence_id" in record
        ]
        processed_sentences = sample_df[sample_df["index"].isin(processed_ids)]
        print(
            f"Number of sentences in sample_df that have already been processed: {len(processed_sentences)}"
        )
        print("Sample of processed sentences:")
        display(sample_df[sample_df["index"].isin(processed_ids)].head(10))
else:
    print("No output file found, so no sentences have been processed yet.")

# Now remove all the already processed sentences from the output file that are not in the sample_df
# if output_file.exists() and output_file.stat().st_size > 0:
#     with open(output_file, "r", encoding="utf-8") as f:
#         processed_data = json.load(f)
#         processed_ids = [
#             int(record["sentence_id"])
#             for record in processed_data
#             if "sentence_id" in record
#         ]
#         filtered_data = [
#             record
#             for record in processed_data
#             if int(record.get("sentence_id", -1)) in sample_df["index"].values
#         ]
#     with open(output_file, "w", encoding="utf-8") as f:
#         json.dump(filtered_data, f, ensure_ascii=False, indent=2)

# Find cumulative total cost from the output file
total_cost = 0.0
if total_cost_txt.exists() and total_cost_txt.stat().st_size > 0:
    with open(total_cost_txt, "r", encoding="utf-8") as f:
        total_cost = float(f.read().strip())
        print(f"Cumulative total cost from log file: ${total_cost:.6f}")
else:
    print("No log file found, so cumulative cost is $0.00.")

Total unprocessed sentences to process: 1523
Next sentence to process:


,num,inflection_type,sentence,word,word_span,label,source,index
6363,2,17,"Ei saa hästi aru, miks oli vaja Tallinnas osatada Hansapanka või Briti nõukogu.",aru,"[13, 16]",[sg p],infl_type_17_1000_v2.json,6363


Number of sentences in sample_df that have already been processed: 6363
Sample of processed sentences:


,num,inflection_type,sentence,word,word_span,label,source,index
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,0
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,1
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,2
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,3
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,4
5,1,1,"Žüriis olid Ave Needo (Ruut FM), Aivo Tammela (Jüri noortemaja), Thea Tekkel (politsei esindaja), Margus Sõna (DJ School) ja Mario Liimann (OÜ Audiomeedia).",Mario,"[125, 130]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,5
6,1,1,"Samas teeb aga murelikuks asjaolu, et paljudes Araabia riikides kasvab rahva seas pahameel oma nn liiga Läänemeelse valitsuse vastu.",Araabia,"[47, 54]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,6
7,1,1,"Meri kinnitas eile Hamburgi linnapeale, et Eesti on valmis viisata suhtluseks kümmet Euroopa Liidu riiki ühendava Schengeni ühisviisaruumiga, sest Eesti piirikaitse on eeskujulik ja üks tänapäevasemaid Euroopas.",Euroopa,"[85, 92]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,7
8,1,1,"Siiditee John W. Orrison, endine CSX Transportationi presidendi abi, on rääkinud Wall Street Journalile, et kaupade eksportimine Hiinast ja Koreast Euroopa Liitu on piiramatu potentsiaaliga äri, kus liigub aastas kümneid miljardeid eurosid.",piiramatu,"[165, 174]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,8
9,1,1,Rahvusvaheline diplomaatia hoidis ära kahe naabri vahelise sõja.,diplomaatia,"[15, 26]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,9


Cumulative total cost from log file: $7.598968


In [22]:
# Batch rewrite loop: call OpenAI and append each result to a JSON file
processed = len(processed_sentences)
max_cost = 20.0  # Set a maximum total cost threshold for the entire batch to avoid unexpected high costs during testing
with tqdm.tqdm(
    total=len(sample_df), desc="Processing sentences", initial=processed
) as pbar:
    for i, row in enumerate(
        zip(
            unprocessed_df["index"],
            unprocessed_df["sentence"],
            unprocessed_df["word_span"],
            unprocessed_df["word"],
            unprocessed_df["label"],
        )
    ):
        sentence_id, sentence, word_span, word, label = row
        try:
            metadata = {
                "target_word": word,
                "label": label.tolist(),  # Convert numpy array to list for JSON serialization
            }
            result, response = rewrite_with_retry(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                config=config,
                metadata=metadata,
            )
            append_result_to_json(output_file, result, id=sentence_id)
            # append_result_to_json(output_file_log, response, id=sentence_id) # Log the full response for debugging and analysis
            # Get the cost info from the result for logging
            cost_info = response.get("cost", {})
            if "error" in cost_info:
                tqdm.tqdm.write(
                    f"[{processed + 1}] Saved sentence_id={sentence_id} with cost estimation error: {cost_info['error']}"
                )
            else:
                total_cost_info = cost_info.get("total_cost", 0.0)
                total_cost += float(total_cost_info)
                write_total_cost(total_cost_txt, total_cost)
                # tqdm.tqdm.write(
                #     f"[{processed + 1}] Saved sentence_id={sentence_id} cost: {total_cost_info} total: {total_cost:.6f}"
                # )
        except Exception as exc:
            error_record = {
                "sentence_id": str(sentence_id),
                "source_sentence": sentence,
                "target_word": word,
                "word_span": word_span.astype(int).tolist(),
                "label": label.tolist(),
                "error": str(exc),
            }
            append_result_to_json(output_file, error_record, id=sentence_id)
            # append_result_to_json(
            #     output_file_log, {"error": str(exc)}, id=sentence_id
            # )  # Log the error details in the log file as well
            tqdm.tqdm.write(
                f"[{processed + 1}] Error on sentence_id={sentence_id}: {exc}"
            )
            print("Traceback:", exc.__traceback__)

        processed += 1
        # update postfix shown to the right of the bar
        pbar.set_postfix({"total_cost": f"${total_cost:.6f}", "processed": processed})
        pbar.update(1)

        if (
            total_cost > max_cost
        ):  # Stop the loop if total cost exceeds $20 to avoid unexpected high costs during testing
            print(
                f"Total cost exceeded ${max_cost}. Stopping the loop to avoid unexpected high costs during testing."
            )
            break

        # Additional pacing between successful/failed items to avoid bursty traffic.
        if i < len(unprocessed_df) - 1:  # Don't sleep after the last item
            sleep_s = config["base_delay_seconds"] + random.uniform(
                0.0, config["jitter_seconds"]
            )
            pbar.set_description_str(f"Sleeping {sleep_s:.2f}s")
            time.sleep(sleep_s)

print(f"Batch processing completed. Total cost: ${total_cost:.6f}")
print(f"Finished. Processed {processed} rows. Results appended to: {output_file}")

Processing sentences:  81%|████████  | 6363/7886 [00:01<?, ?it/s]

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/EstNLTK-gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Sleeping 0.09s:  81%|████████  | 6364/7886 [00:02<34:00,  1.34s/it, total_cost=$7.600146, processed=6364]      

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/EstNLTK-gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Sleeping 0.10s:  81%|████████  | 6365/7886 [00:03<29:53,  1.18s/it, total_cost=$7.601343, processed=6365]

...

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/EstNLTK-gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Sleeping 0.13s: 100%|█████████▉| 7885/7886 [35:09<00:01,  1.90s/it, total_cost=$9.438723, processed=7885]

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/EstNLTK-gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Sleeping 0.13s: 100%|██████████| 7886/7886 [35:10<00:00,  1.39s/it, total_cost=$9.439898, processed=7886]

Batch processing completed. Total cost: $9.439898
Finished. Processed 7886 rows. Results appended to: ..\outputs\gpt-4o-homonyms_LLM_MLM.json


In [23]:
# Open LLM sample annotation data
llm_samples_df = pd.read_json(output_file, orient="records")

new_records = []

# For each record, iterate over the candidates, construct the candidate sentence, and perform morphological analysis to extract the case and other features of the candidate replacement word. Assume each candidate has uniform probability and sum up the probabilities for candidates that match each form label to get a distribution of form labels for the candidates.
for index, row in tqdm.tqdm(llm_samples_df.iterrows(), total=llm_samples_df.shape[0]):
    id = row["sentence_id"]
    candidates = row.get("candidates", [])
    source_sentence = row["source_sentence"]
    target_word = row["target_word"]
    word_span = row["word_span"]
    label = row.get("label", [])[
        0
    ]  # Assuming label is a list and we take the first element as the form label
    # print(f"Source sentence: {source_sentence}")
    # print(f"Target word span: {word_span}")
    candidate_form_weight = (
        1 / len(candidates) if candidates else 0
    )  # Uniform weight for each candidate
    form_probabilities = {}
    for candidate in candidates:
        # Construct sentence with the candidate replacement
        candidate_sentence = (
            source_sentence[: word_span[0]]
            + candidate
            + source_sentence[word_span[1] :]
        )
        # print(f"Candidate sentence: {candidate_sentence}")
        # Create EstNLTK Text object and perform morphological analysis
        estnltk_text = estnltk.Text(candidate_sentence)
        estnltk_text.tag_layer("morph_analysis")
        # Extract the morphological tags for the candidate word
        morph_layer = estnltk_text.morph_analysis
        # Find the token in the morph layer that corresponds to the candidate replacement
        candidate_token = None
        for token in morph_layer:
            if (
                token.start <= word_span[0] < token.end
            ):  # Check if the token covers the start of the original word span
                candidate_token = token
                break
        if candidate_token:
            # Count up the form labels for the candidate
            number_of_labels = len(candidate_token.form) if candidate_token.form else 0
            if number_of_labels > 0:
                weight_per_label = candidate_form_weight / number_of_labels
                for candidate_label in candidate_token.form:
                    form_probabilities[candidate_label] = (
                        form_probabilities.get(candidate_label, 0) + weight_per_label
                    )
            else:
                form_probabilities["unknown"] = (
                    form_probabilities.get("unknown", 0) + candidate_form_weight
                )

    best_form_label = (
        max(form_probabilities, key=form_probabilities.get)
        if form_probabilities
        else "unknown"
    )
    # Create a record for the chosen candidate and its features
    candidate_record = {
        "id": id,
        "candidates": candidates,
        "pred_label": best_form_label,
        "true_label": label,
        "predicted_form_distribution": form_probabilities,
        "source_sentence": source_sentence,
        "target_word": target_word,
        "word_span": word_span,
    }
    new_records.append(candidate_record)

100%|██████████| 7886/7886 [07:52<00:00, 16.67it/s]


In [24]:
# Create a new DataFrame from the predicted form labels and their distributions and save to JSON
homonym_results_df = pd.DataFrame(new_records)
homonym_results_df_path = (
    HOMONYMS_DIRS["processed"] / "homonyms_llm_mlm_predictions_full.parquet"
)
homonym_results_df.to_parquet(homonym_results_df_path, index=False)

In [25]:
client.close()

In [ ]:
# previous_created = -1
# if output_file_log.exists() and output_file.exists():
#     with open(output_file_log, "r", encoding="utf-8") as f:
#         with open(output_file, "r", encoding="utf-8") as f2:
#             # Check if each record in output has logs
#             # Check via candidate list
#             output_data = json.load(f2)
#             log_data = json.load(f)
#             output_candidates = {}
#             log_candidates = {}
#             for record in output_data:
#                 sentence_id = int(record.get("sentence_id", -1))
#                 candidates = record.get("candidates", [])
#                 output_candidates[sentence_id] = candidates
#             for i, log_record in enumerate(log_data):
#                 candidates = log_record.get("choices")[0].get("message").get("content")
#                 # Parse the candidates from the log content
#                 candidate_data = json.loads(candidates)
#                 parsed_candidates = candidate_data.get("candidates")
#                 log_candidates[i] = parsed_candidates
#             for i in range(max(len(output_candidates), len(log_candidates))):
#                 output_cands = output_candidates.get(i)
#                 log_cands = log_candidates.get(i)
#                 if output_cands != log_cands:
#                     print(f"Mismatch in candidates for sentence_id={i}:")
#                     print(f"Output file candidates: {output_cands}")
#                     print(f"Log file candidates: {log_cands}")
#                     break

Mismatch in candidates for sentence_id=3710:
Output file candidates: ['Wiesenthali', 'Schindleri', 'Goldbergi', 'Edelsteini', 'Kaufmani', 'Liebmanni', 'Rosenbergi', 'Mendelsohni', 'Schwarzbergi', 'Leibovitsi']
Log file candidates: ['toimet', 'tagajärge', 'mõjust', 'tulemit', 'mõjuvõimu', 'tagasilööki', 'efekti', 'mõjujõudu', 'koormust', 'kõrvalnähtu']


In [ ]:
# # Estimate the total cost of the already processed sentences
# total_cost = 0.0  # Reset total cost for testing purposes
# enc = tiktoken.encoding_for_model(model_name)  # pick same model you'll use
# if output_file.exists() and output_file.stat().st_size > 0:
#     with open(output_file, "r", encoding="utf-8") as f:
#         data = json.load(f)
#         for row in data:
#             source_sentence = row.get("source_sentence", "")
#             word_span = row.get("word_span", [0, 0])
#             user_prompt = create_user_prompt(
#                 sentence=source_sentence, word_span=word_span
#             )
#             messages = build_messages(user_prompt)
#             # Calculate prompt tokens
#             # input cost
#             prompt_text = "".join(m["content"] for m in messages)
#             prompt_tokens = len(enc.encode(prompt_text)) + 30
#             input_cost = costs[model_name]["input"] * prompt_tokens
#             # cached input cost (not applicable in this case, but included for completeness)
#             cached_input_cost = 0.0
#             # output cost
#             prompt_text = '{"candidates":' + str(row.get("candidates", [])) + "}"
#             output_tokens = len(enc.encode(prompt_text)) - 8
#             output_cost = costs[model_name]["output"] * output_tokens
#             total_cost += input_cost + cached_input_cost + output_cost
#             # print(
#             #     f"Token counts for sentence_id={row.get('sentence_id', 'N/A')}: prompt_tokens={prompt_tokens}, output_tokens={output_tokens}"
#             # )
#             # print(
#             #     f"Costs for sentence_id={row.get('sentence_id', 'N/A')}: input_cost=${input_cost:.6f}, cached_input_cost=${cached_input_cost:.6f}, output_cost=${output_cost:.6f}"
#             # )
#             # print(
#             #     f"Estimated cost for sentence_id={row.get('sentence_id', 'N/A')}: ${input_cost + cached_input_cost + output_cost:.6f}"
#             # )
# print(f"Total estimated cost for already processed sentences: ${total_cost:.6f}")

Total estimated cost for already processed sentences: $7.299983


### BERT-LLM-morph Batch request


In [3]:
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

In [4]:
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to select the best 5 candidates from the provided 25 BERT candidates to replace the marked word <...> in the sentence.

Rules:
- Do not invent new candidates.
- Preserve the word's tense, number, case agreement, punctuation, and capitalisation.
- Return candidates that fit naturally and grammatically in one of these cases: nominative, genitive, partitive, or additive/illative.
- For proper names, keep only proper-name candidates that fit the same grammatical context.
- If fewer than 5 valid candidates exist, return as many as possible.
- Order candidates from best to worst fit.
- Do not repeat the original word unless it is the only valid option.
"""

# Structured output format for Azure OpenAI chat completions.
# Structured Outputs require a top-level object.
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "bert_llm_selection",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "candidates": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 1,
                    "maxItems": 5,
                },
            },
            "required": ["candidates"],
            "additionalProperties": False,
        },
    },
}

# Few-shot example to anchor the output format
few_shot_user = """Sentence: "Ehk oleks mõttekas ka mõni selleteemaline hoiatav <kampaania> korraldada," lisab punase autoga preili.
Candidates: ['aktsioon', 'üritus', 'kampaania', 'sündmus', 'saade', 'koosolek', 'kõne', 'päev', 'postitus', 'artikkel', 'video', 'näitus', 'asi', 'lugu', 'teade', 'lugemine', 'leht', 'liiklus', 'hoiatus', 'uudis', 'post', 'teavitus', 'loeng', 'komm', 'akt']
"""

few_shot_assistant = json.dumps(
    {
        "candidates": [
            "aktsioon",
            "üritus",
            "sündmus",
            "näitus",
            "teade",
        ],
    },
    ensure_ascii=False,
    indent=2,
)


def filter_candidates(candidates: list[str]) -> list[str]:
    """Filter out candidates that are likely to be jailbreaks or otherwise invalid based on simple heuristics."""
    filtered = []
    for candidate in candidates:
        if not isinstance(candidate, str):
            continue  # Skip non-string candidates
        candidate_lower = candidate.lower()
        if any(
            marker in candidate_lower
            for marker in [
                "<s>",
                "</s>",
                "<pad>",
                "<unk>",
                "NOTUSED",
                "INVALID",
                "JAILBREAK",
                "HACK",
                "EXPLOIT",
            ]
        ):
            continue  # Skip candidates with suspicious markers
        if re.search(r"[<>/\\{}]", candidate):
            # Skip candidates containing special characters that could indicate an attempt to break the format or inject content
            continue
        if not any(ch.isalpha() for ch in candidate):
            # Skip candidates that don't contain any alphabetic characters, as they are unlikely to be valid word replacements
            continue
        filtered.append(candidate)
    return filtered


# Create a user prompt template for the real sentences, including candidates list.
def create_user_prompt(sentence: str, word_span: tuple, candidates: list[str]) -> str:
    """Create a user prompt with marked sentence and candidate list."""
    # Mark the target word with angle brackets
    start, end = word_span
    marked_sentence = (
        sentence[:start] + "<" + sentence[start:end] + ">" + sentence[end:]
    )
    # Filter jailbreak candidates
    candidates = filter_candidates(candidates)
    candidates_str = str(candidates)  # Simple list representation
    return f"""Sentence: {marked_sentence}
Candidates: {candidates_str}
"""


def build_messages(user_prompt: str) -> list[dict[str, str]]:
    """Build the chat-completions message list for one rewrite request."""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": few_shot_user},
        {"role": "assistant", "content": few_shot_assistant},
        {"role": "user", "content": user_prompt},
    ]


def parse_response_content(content: str) -> dict[str, Any]:
    """Parse and validate the JSON payload returned by Azure OpenAI."""
    parsed = json.loads(content or "{}")
    if not isinstance(parsed, dict):
        raise ValueError("Expected a JSON object from the model response.")
    if "candidates" not in parsed or not isinstance(parsed["candidates"], list):
        raise ValueError("Model response did not contain a candidates list.")
    return parsed


def _atomic_write_json(file_path: pathlib.Path, data: list[dict[str, Any]]) -> None:
    """Write JSON atomically to avoid partial files on crash/interruption."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    temp_file_path: pathlib.Path | None = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=file_path.parent,
            prefix=f"{file_path.stem}.",
            suffix=".tmp",
            delete=False,
        ) as temp_file:
            json.dump(data, temp_file, ensure_ascii=False, indent=2)
            temp_file.flush()
            os.fsync(temp_file.fileno())
            temp_file_path = pathlib.Path(temp_file.name)

        # Atomic replace: after a crash you see either the old or the new file, never a partial one.
        os.replace(temp_file_path, file_path)
    finally:
        if temp_file_path is not None and temp_file_path.exists():
            temp_file_path.unlink(missing_ok=True)


def _load_results_json(file_path: pathlib.Path) -> list[dict[str, Any]]:
    """Load results JSON safely and fail closed on corruption to prevent silent data loss."""
    if not file_path.exists() or file_path.stat().st_size == 0:
        return []

    with open(file_path, "r", encoding="utf-8") as file_handle:
        try:
            loaded = json.load(file_handle)
        except json.JSONDecodeError as error:
            raise RuntimeError(
                f"Failed to parse existing output file {file_path}. "
                "Refusing to overwrite it to avoid data loss."
            ) from error

    if not isinstance(loaded, list):
        raise RuntimeError(
            f"Expected a JSON list in {file_path}, got {type(loaded).__name__}. "
            "Refusing to overwrite it to avoid data loss."
        )
    return loaded


def append_result_to_json(
    file_path: pathlib.Path, record: dict[str, Any], id: str | None = None
) -> None:
    """Append or update one record in a JSON array using crash-safe atomic writes."""
    resolved_path = pathlib.Path(file_path)
    data = _load_results_json(resolved_path)

    if id is not None:
        # If ID is provided, update matching record; otherwise append.
        existing_record = next(
            (
                r
                for r in data
                if isinstance(r, dict)
                and r.get("id") is not None
                and str(r.get("id")) == str(id)
            ),
            None,
        )
        if existing_record:
            existing_record.update(record)
            # Remove stale error after a successful rewrite.
            existing_record.pop("error", None)
        else:
            data.append(record)
    else:
        data.append(record)

    _atomic_write_json(resolved_path, data)


def write_total_cost(file_path: pathlib.Path, total_cost: float) -> None:
    """Write the total cost to a separate txt file."""
    resolved_path = pathlib.Path(file_path)
    resolved_path.parent.mkdir(parents=True, exist_ok=True)
    with open(resolved_path, "w", encoding="utf-8") as f:
        f.write(str(total_cost))


def rewrite_sentence_with_azure_openai(
    sentence_id: int,
    sentence: str,
    word_span: tuple,
    candidate_details: list[dict[str, Any]],
    metadata: dict[str, Any],
    candidates: list[str],
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Select best 5 BERT candidates via Azure OpenAI and derive predicted label from candidate scores."""
    user_prompt = create_user_prompt(
        sentence=sentence, word_span=word_span, candidates=candidates
    )
    messages = build_messages(user_prompt)

    # Request a structured JSON response from Azure OpenAI chat completions.
    response = client.chat.completions.create(
        model=deployment_name,
        messages=messages,
        response_format=response_format,
        max_completion_tokens=512,
        top_p=0.95,
    )

    # Estimate cost for this response (best-effort).
    try:
        cost_result = estimate_cost_typed(response)
        cost_info = cost_result.as_dict()  # Convert to dict for JSON serialization
    except Exception as _err:
        # Don't fail the rewrite on estimator errors; keep a surfacable error message.
        cost_info = {"error": str(_err)}

    message = response.choices[0].message
    refusal = getattr(message, "refusal", None)
    if refusal:
        raise RuntimeError(f"Model refused the request: {refusal}")

    # Parse the JSON response and enrich it with metadata for later analysis.
    parsed = parse_response_content(message.content or "{}")
    response_dict = response.to_dict()

    # Extract selected candidates and process BERT candidate details.
    selected_candidates = parsed.get("candidates", [])
    if not isinstance(selected_candidates, list):
        selected_candidates = []

    selected_candidate_details = [
        candidate
        for candidate in candidate_details
        if candidate.get("token") in selected_candidates
    ]

    # Normalise scores across selected candidates.
    selected_score_total = sum(
        candidate.get("score", 0.0) for candidate in selected_candidate_details
    )
    for candidate in selected_candidate_details:
        candidate_score = candidate.get("score", 0.0)
        candidate["normalised_score"] = (
            candidate_score / selected_score_total if selected_score_total > 0 else 0.0
        )

    # Derive morphological label predictions from selected candidate scores.
    predictions = {}
    for candidate in selected_candidate_details:
        resolved_form = candidate.get("resolved_form", [])
        if not resolved_form:
            continue
        label = resolved_form[0]
        predictions[label] = predictions.get(label, 0.0) + candidate["normalised_score"]

    predicted_label = max(predictions, key=predictions.get) if predictions else None

    # Enrich parsed response with metadata and analysis results.
    parsed["sentence_id"] = sentence_id
    parsed["candidate_details"] = selected_candidate_details
    parsed["predictions"] = predictions
    parsed["pred_label"] = predicted_label
    parsed["source_sentence"] = sentence
    parsed["target_word"] = metadata.get("target_word", "")

    if isinstance(word_span, list):
        parsed["word_span"] = word_span
    else:
        parsed["word_span"] = word_span.astype(
            int
        ).tolist()  # Convert numpy array to list for JSON serialization

    parsed["label"] = metadata.get("label", [])

    # Attach cost estimation metadata.
    response_dict["cost"] = cost_info

    return parsed, response_dict


# Backwards-compatible alias for any existing callers.
rewrite_sentence_with_genai = rewrite_sentence_with_azure_openai


def get_latest_processed_id(output_file):
    """Get the highest sentence ID that has already been processed and saved in the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        with open(output_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                return max(int(record["id"]) for record in data if "id" in record)
    return -1  # No valid records found, start from the beginning


def get_unprocessed_dataset(dataset, output_file):
    """Filter the dataset to include only records that have errors or have not been processed yet, based on the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        unprocessed_records = []
        processed_records = []
        # First, get the list of sentences that have errors in the output file
        with open(output_file, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
            for record in processed_data:
                if "error" in record:
                    unprocessed_records.append(record["source_sentence"])
                else:
                    processed_records.append(record["source_sentence"])
        # Now filter the original dataset to include only records that are either unprocessed or have errors
        filtered_dataset = dataset[
            dataset["sentence"].isin(unprocessed_records)
            | ~dataset["sentence"].isin(processed_records)
        ]
        return filtered_dataset
    return dataset  # If no output file exists, return the entire dataset for processing


def is_transient_error(exc: Exception) -> bool:
    """Heuristically detect retryable errors such as rate limit and network hiccups."""
    message = str(exc).lower()
    transient_markers = [
        "429",
        "rate",
        "quota",
        "too many requests",
        "timeout",
        "timed out",
        "connection",
        "temporar",
        "unavailable",
        "503",
        "502",
        "504",
        "internal",
    ]
    return any(marker in message for marker in transient_markers)


def rewrite_with_retry(
    sentence_id: int,
    sentence: str,
    word_span: tuple,
    candidate_details: list[dict[str, Any]],
    metadata: dict[str, Any],
    candidates: list[str],
    config: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Call rewrite function with exponential backoff on transient errors."""
    for attempt in range(1, config["max_attempts"] + 1):
        try:
            return rewrite_sentence_with_azure_openai(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                candidate_details=candidate_details,
                metadata=metadata,
                candidates=candidates,
            )
        except Exception as exc:
            should_retry = is_transient_error(exc) and attempt < config["max_attempts"]
            # should_retry = (
            #     attempt < config["max_attempts"]
            # )  # Retry on all exceptions for robustness, but still limit the number of attempts
            if not should_retry:
                raise
            backoff = min(
                config["max_backoff_seconds"],
                config["initial_backoff_seconds"] * (2 ** (attempt - 1)),
            )
            sleep_s = backoff + random.uniform(0.0, config["jitter_seconds"])
            print(
                f"Retry {attempt}/{config['max_attempts'] - 1} for sentence_id={sentence_id} "
                f"after transient error: {exc}. Sleeping {sleep_s:.2f}s..."
            )
            time.sleep(sleep_s)

In [5]:
bert_mlm_candidates_filepath = (
    HOMONYMS_DIRS["processed"] / "homonyms_bert_mlm_candidate_details_100.jsonl"
)
with open(bert_mlm_candidates_filepath, "r", encoding="utf-8") as f:
    bert_mlm_candidates_data = [json.loads(line) for line in f]

bert_mlm_candidates_df = pd.DataFrame(bert_mlm_candidates_data)

# Rename the "true_label" column to "label" to stay consistent with the rest of the code
bert_mlm_candidates_df.rename(columns={"true_label": "label"}, inplace=True)

display(bert_mlm_candidates_df.head(1))

row_index  inflection_type  \
0          0                1   

                                                                                                                                                                                                              sentence  \
0  Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.   

     word word_span label predicted_best_label  predicted_best_label_score  \
0  võitja  [74, 80]  sg n                 sg n                     0.72215   

                                                                                                                                                                                                                                      predicted_top_labels  \
0  [{'label': 'sg n', 'score': 0.7221502446191154}, {'label': 'sg g', 'score': 0.18655089150706772}, {'label': 'pl n', 'score': 0.031669486401369795}, {'label': 'nud', 'score': 0.03149387186567765}, {'label': 'sg tr', 'score': 0.0014702172484248877}]   

                                                                                                                                                                                                    label_scores  \
0  {'sg n': 0.7221502446191154, 'sg g': 0.18655089150706772, 'nud': 0.03149387186567765, 'pl n': 0.031669486401369795, 'sg tr': 0.0014702172484248877, 'o': 0.00035745027707889676, 's': 0.00035488014691509306}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [9]:
output_file = (
    OUTPUT_DIR / "gpt-4o-homonyms_BERT_MLM_to_LLM.json"
)  # Output file for results
output_file_log = (
    OUTPUT_DIR / "gpt-4o-homonyms_BERT_MLM_to_LLM_log.json"
)  # More detailed log file for debugging and analysis
total_cost_txt = (
    OUTPUT_DIR / "gpt-4o-homonyms_BERT_MLM_to_LLM_total_cost.txt"
)  # File to store the total cost information
sample_df = bert_mlm_candidates_df.iloc[:].copy()  # Use N for testing
sample_df["index"] = sample_df.index.astype(
    int
)  # Ensure index is an integer for ID purposes

# gpt-4o has following rate limits:
# - 180 000 tokens per minute (TPM)
# - 1080 requests per minute (RPM)
config = {
    # Pacing configuration to avoid hitting rate limits
    "base_delay_seconds": round(60 / 1080, 2),  # RPM is 1080 for gpt-4o
    "jitter_seconds": 0.1,  # Add a small random jitter to avoid thundering herd issues
    # Retry configuration for transient failures
    "max_attempts": 6,
    "initial_backoff_seconds": 2.0,
    "max_backoff_seconds": 60.0,
}

# Start from the next unprocessed sentence based on the output file contents
unprocessed_df = get_unprocessed_dataset(sample_df, output_file)
if len(unprocessed_df) == 0:
    print("No unprocessed sentences found. All done!")
else:
    print(f"Total unprocessed sentences to process: {len(unprocessed_df)}")
    print("Next sentence to process:")
    display(unprocessed_df.head(1))

# Find processed records that are in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["sentence_id"])
            for record in processed_data
            if "sentence_id" in record
        ]
        processed_sentences = sample_df[sample_df["index"].isin(processed_ids)]
        print(
            f"Number of sentences in sample_df that have already been processed: {len(processed_sentences)}"
        )
        print("Sample of processed sentences:")
        display(sample_df[sample_df["index"].isin(processed_ids)].head(10))
else:
    processed_sentences = (
        pd.DataFrame()
    )  # Default to empty DataFrame if no output file exists
    print("No output file found, so no sentences have been processed yet.")

# Now remove all the already processed sentences from the output file that are not in the sample_df
# if output_file.exists() and output_file.stat().st_size > 0:
#     with open(output_file, "r", encoding="utf-8") as f:
#         processed_data = json.load(f)
#         processed_ids = [
#             int(record["sentence_id"])
#             for record in processed_data
#             if "sentence_id" in record
#         ]
#         filtered_data = [
#             record
#             for record in processed_data
#             if int(record.get("sentence_id", -1)) in sample_df["index"].values
#         ]
#     with open(output_file, "w", encoding="utf-8") as f:
#         json.dump(filtered_data, f, ensure_ascii=False, indent=2)

# Find cumulative total cost from the output file
total_cost = 0.0
if total_cost_txt.exists() and total_cost_txt.stat().st_size > 0:
    with open(total_cost_txt, "r", encoding="utf-8") as f:
        total_cost = float(f.read().strip())
        print(f"Cumulative total cost from log file: ${total_cost:.6f}")
else:
    print("No log file found, so cumulative cost is $0.00.")

Total unprocessed sentences to process: 2
Next sentence to process:


row_index  inflection_type  \
296        296                1   

                                                                                                                         sentence  \
296  (Algus lk. 5) "Võin teile vastata sõnaga, mida kasutan intervjuud andes väga harva : (teie poolt öeldu on) täielik nonsenss!   

      word  word_span label predicted_best_label  predicted_best_label_score  \
296  öeldu  [97, 102]  sg n                 sg n                     0.30574   

                                                                                                                                                                                                                                predicted_top_labels  \
296  [{'label': 'sg n', 'score': 0.3057395956129767}, {'label': 'pl n', 'score': 0.11585979379015043}, {'label': 'tud', 'score': 0.10208245331887156}, {'label': 'b', 'score': 0.04203795513603836}, {'label': 'vad', 'score': 0.03401518240571022}]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   label_scores  \
296  {'pl n': 0.11585979379015043, 'sg n': 0.3057395956129767, 'tud': 0.10208245331887156, 'b': 0.04203795513603836, 'vad': 0.03401518240571022, 'takse': 0.02752561680972576, 'des': 0.023742632009088993, 'sg es': 0.00836722832173109, 'ti': 0.007855487056076527, 'da': 0.009040222270414233, 'sg g': 0.0034738772083073854, 'sg tr': 0.0026254456024616957, 's': 0.004288185737095773, 'neg o': 0.0022131989244371653, 'neg': 0.0015161627670750022, 'n': 0.0013502751244232059, 'me': 0.0013301837025210261, 'sg p': 0.00211423949804157}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

Number of sentences in sample_df that have already been processed: 7884
Sample of processed sentences:


row_index  inflection_type  \
0          0                1   
1          1                1   
2          2                1   
3          3                1   
4          4                1   
5          5                1   
6          6                1   
7          7                1   
8          8                1   
9          9                1   

                                                                                                                                                                                                                                           sentence  \
0                               Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.   
1                                                                                                            Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.   
2                                                                                                                                             "Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada," lisab punase autoga preili.   
3                                                                                   "Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha," kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.   
4                                                                               Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea "suur tarkuseavaldus".   
5                                                                                      Žüriis olid Ave Needo (Ruut FM), Aivo Tammela (Jüri noortemaja), Thea Tekkel (politsei esindaja), Margus Sõna (DJ School) ja Mario Liimann (OÜ Audiomeedia).   
6                                                                                                              Samas teeb aga murelikuks asjaolu, et paljudes Araabia riikides kasvab rahva seas pahameel oma nn liiga Läänemeelse valitsuse vastu.   
7                               Meri kinnitas eile Hamburgi linnapeale, et Eesti on valmis viisata suhtluseks kümmet Euroopa Liidu riiki ühendava Schengeni ühisviisaruumiga, sest Eesti piirikaitse on eeskujulik ja üks tänapäevasemaid Euroopas.   
8  Siiditee John W. Orrison, endine CSX Transportationi presidendi abi, on rääkinud Wall Street Journalile, et kaupade eksportimine Hiinast ja Koreast Euroopa Liitu on piiramatu potentsiaaliga äri, kus liigub aastas kümneid miljardeid eurosid.   
9                                                                                                                                                                                  Rahvusvaheline diplomaatia hoidis ära kahe naabri vahelise sõja.   

          word   word_span label predicted_best_label  \
0       võitja    [74, 80]  sg n                 sg n   
1      teooria    [20, 27]  sg n                 sg n   
2    kampaania    [51, 60]  sg n                 sg n   
3         õige    [16, 20]  sg n                 sg n   
4      Itaalia      [0, 7]  sg g                 sg g   
5        Mario  [125, 130]  sg n                 sg n   
6      Araabia    [47, 54]  sg g                 sg g   
7      Euroopa    [85, 92]  sg g                 sg g   
8    piiramatu  [165, 174]  sg g                 sg g   
9  diplomaatia    [15, 26]  sg n                 sg n   

   predicted_best_label_score  \
0                    0.722150   
1                    0.835027   
2                    0.925145   
3                    0.937690   
4                    0.633394   
5                    0.731673   
6                    0.804141   
7                    0.999933

Cumulative total cost from log file: $18.724012


In [7]:
create_user_prompt(
    sentence=unprocessed_df.iloc[0]["sentence"],
    word_span=unprocessed_df.iloc[0]["word_span"],
    candidates=[
        c.get("token")
        for c in unprocessed_df.iloc[0]["candidate_details"]
        if "token" in c
    ],
)

'Sentence: (Algus lk. 5) "Võin teile vastata sõnaga, mida kasutan intervjuud andes väga harva : (teie poolt <öeldu> on) täielik nonsenss!\nCandidates: [\'öeldud\', \'see\', \'on\', \'öeldakse\', \'nimetatud\', \'kirjutatud\', \'vastus\', \'öeldes\', \'mis\', \'kasutatud\', \'lause\', \'või\', \'kasutamine\', \'ja\', \'öel\', \'ütlemata\', \'valitud\', \'vaadatuna\', \'väidetud\', \'öeldi\', \'öelda\', \'antud\', \'tähendab\', \'sõna\', \'nimetamine\', \'mõeldud\', \'sõnad\', \'hääl\', \'vale\', \'mainitud\', \'valetamine\', \'esitatud\', \'õige\', \'ka\', \'olemine\', \'siis\', \'lisatud\', \'märgitud\', \'aga\', \'pöördumine\', \'väljendatud\', \'nagu\', \'kuidas\', \'lihtsalt\', \'vastuseks\', \'oli\', \'tehtud\', \'kasutades\', \'vastates\', \'pole\', \'kirjas\', \'et\', \'kirjutamine\', \'kirjutades\', \'avaldatud\', \'viisakas\', \'tsitaat\', \'siin\', \'ütles\', \'küsitud\', \'kas\', \'ei\', \'olev\', \'ta\', \'kes\', \'kasutatav\', \'kui\', \'alati\', \'ütlen\', \'vaadates\', \'

In [8]:
# Batch rewrite loop: call OpenAI and append each result to a JSON file
processed = len(processed_sentences)
max_cost = 20.0  # Set a maximum total cost threshold for the entire batch to avoid unexpected high costs during testing
with tqdm.tqdm(
    total=len(sample_df), desc="Processing sentences", initial=processed
) as pbar:
    for i, row in enumerate(
        zip(
            unprocessed_df["index"],
            unprocessed_df["sentence"],
            unprocessed_df["word_span"],
            unprocessed_df["word"],
            unprocessed_df["label"],
            unprocessed_df["candidate_details"],
        )
    ):
        sentence_id, sentence, word_span, word, label, candidate_details = row
        try:
            metadata = {"target_word": word, "label": label}
            candidates = [c.get("token") for c in candidate_details if "token" in c]
            result, response = rewrite_with_retry(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                candidate_details=candidate_details,
                metadata=metadata,
                candidates=candidates,
                config=config,
            )
            # Extract the parsed response (first element of tuple); ignore the response_dict for now
            parsed_result = result[0] if isinstance(result, tuple) else result
            append_result_to_json(output_file, parsed_result, id=sentence_id)
            append_result_to_json(
                output_file_log, response, id=sentence_id
            )  # Log the full response for debugging and analysis
            # Get the cost info from the result for logging
            cost_info = response.get("cost", {})
            if "error" in cost_info:
                tqdm.tqdm.write(
                    f"[{processed + 1}] Saved sentence_id={sentence_id} with cost estimation error: {cost_info['error']}"
                )
            else:
                total_cost_info = cost_info.get("total_cost", 0.0)
                total_cost += float(total_cost_info)
                write_total_cost(total_cost_txt, total_cost)
                # tqdm.tqdm.write(
                #     f"[{processed + 1}] Saved sentence_id={sentence_id} cost: {total_cost_info} total: {total_cost:.6f}"
                # )
        except Exception as exc:
            error_record = {
                "id": str(sentence_id),
                "source_sentence": sentence,
                "target_word": word,
                "word_span": word_span,
                "true_label": [label] if isinstance(label, str) else label.tolist(),
                "pred_label": None,
                "error": str(exc),
            }
            # append_result_to_json(output_file, error_record, id=sentence_id)
            # append_result_to_json(
            #     output_file_log, {"error": str(exc)}, id=sentence_id
            # )  # Log the error details in the log file as well
            tqdm.tqdm.write(
                f"[{processed + 1}] Error on sentence_id={sentence_id}: {exc}"
            )
            print("Error record:", error_record)
            print("Traceback:", exc.__traceback__)

        processed += 1
        # update postfix shown to the right of the bar
        pbar.set_postfix({"total_cost": f"${total_cost:.6f}", "processed": processed})
        pbar.update(1)

        if (
            total_cost > max_cost
        ):  # Stop the loop if total cost exceeds $20 to avoid unexpected high costs during testing
            print(
                f"Total cost exceeded ${max_cost}. Stopping the loop to avoid unexpected high costs during testing."
            )
            break

        # Additional pacing between successful/failed items to avoid bursty traffic.
        if i < len(unprocessed_df) - 1:  # Don't sleep after the last item
            sleep_s = config["base_delay_seconds"] + random.uniform(
                0.0, config["jitter_seconds"]
            )
            pbar.set_description_str(f"Sleeping {sleep_s:.2f}s")
            time.sleep(sleep_s)

print(f"Batch processing completed. Total cost: ${total_cost:.6f}")
print(f"Finished. Processed {processed} rows. Results appended to: {output_file}")

Sleeping 0.10s:  86%|████████▌ | 6762/7886 [00:01<21:32,  1.15s/it, total_cost=$16.007565, processed=6762]      

[6762] Error on sentence_id=296: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'jailbreak': {'detected': True, 'filtered': True}}}}}
Error record: {'id': '296', 'source_sentence': '(Algus lk. 5) "Võin teile vastata sõnaga, mida kasutan intervjuud andes väga harva : (teie poolt öeldu on) täielik nonsenss!', 'target_word': 'öeldu', 'word_span': [97, 102], 'true_label': ['sg n'], 'pred_label': None, 'error': 'Error code: 400 - {\'error\': {\'message\': "The response was filtered due to the prompt triggering Azure OpenAI\'s content management policy. Please modify your prompt and r

Sleeping 0.06s:  86%|████████▌ | 6763/7886 [00:01<15:27,  1.21it/s, total_cost=$16.007565, processed=6763]

[6763] Error on sentence_id=668: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'jailbreak': {'detected': True, 'filtered': True}}}}}
Error record: {'id': '668', 'source_sentence': 'Sel juhul väldid olukorda, kus saad dünamo.', 'target_word': 'dünamo', 'word_span': [36, 42], 'true_label': ['sg g'], 'pred_label': None, 'error': 'Error code: 400 - {\'error\': {\'message\': "The response was filtered due to the prompt triggering Azure OpenAI\'s content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our document

Sleeping 0.12s: 100%|██████████| 7886/7886 [1:15:23<00:00,  4.02s/it, total_cost=$18.724012, processed=7886]

Batch processing completed. Total cost: $18.724012
Finished. Processed 7886 rows. Results appended to: C:\Users\sande\Git_projects\EstNLTK\EstNLTK_model_training\outputs\gpt-4o-homonyms_BERT_MLM_to_LLM.json


In [10]:
# Open LLM annotation data
llm_samples_df = pd.read_json(output_file, orient="records")

In [11]:
display(llm_samples_df.head(1))

,candidates,sentence_id,candidate_details,predictions,pred_label,source_sentence,target_word,word_span,label
0,"[võitnud, saaja, pälvinud, auhinna, tiitli]",0,"[{'rank': 3, 'token': 'auhinna', 'score': 0.056165203452110006, 'sequence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri auhinna James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_sentence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri auhinna James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_span': [74, 81], 'resolved_forms': ['sg g'], 'normalised_score': 0.43247800253933105}, {'rank': 4, 'token': 'võitnud', 'score': 0.05125705897808001, 'sequence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitnud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_sentence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitnud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_span': [74, 81], 'resolved_forms': ['nud', 'pl n', 'sg n'], 'normalised_score': 0.394684771359947}, {'rank': 7, 'token': 'pälvinud', 'score': 0.015388736501336, 'sequence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri pälvinud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_sentence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri pälvinud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_span': [74, 82], 'resolved_forms': ['nud', 'pl n', 'sg n'], 'normalised_score': 0.118494897456868}, {'rank': 9, 'token': 'saaja', 'score': 0.006725015118718001, 'sequence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri saaja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_sentence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri saaja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_span': [74, 79], 'resolved_forms': ['sg g'], 'normalised_score': 0.051783327163942}, {'rank': 58, 'token': 'tiitli', 'score': 0.000332333293044, 'sequence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri tiitli James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_sentence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri tiitli James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.', 'candidate_span': [74, 80], 'resolved_forms': ['sg g'], 'normalised_score': 0.002559001479909}]",{},NaN,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",sg n


In [12]:
display(llm_samples_df.iloc[0]["candidate_details"])

[{'rank': 3,
  'token': 'auhinna',
  'score': 0.056165203452110006,
  'sequence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri auhinna James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.',
  'candidate_sentence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri auhinna James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.',
  'candidate_span': [74, 81],
  'resolved_forms': ['sg g'],
  'normalised_score': 0.43247800253933105},
 {'rank': 4,
  'token': 'võitnud',
  'score': 0.05125705897808001,
  'sequence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitnud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.',
  'candidate_sentence': 'Edinburghi agulite mehe Irvine Welsh

In [14]:
new_records = []

for index, row in tqdm.tqdm(llm_samples_df.iterrows(), total=llm_samples_df.shape[0]):
    id = row["sentence_id"]
    candidates = row.get("candidates", [])
    candidate_details = row.get("candidate_details", [])
    source_sentence = row["source_sentence"]
    target_word = row["target_word"]
    word_span = row["word_span"]
    label = row.get("label", [])[
        0
    ]  # Assuming label is a list and we take the first element as the form label
    # print(f"Source sentence: {source_sentence}")
    # print(f"Target word span: {word_span}")
    form_probabilities = {}
    for candidate in candidate_details:
        candidate_sentence = candidate.get("candidate_sentence")
        candidate_span = candidate.get("candidate_span", word_span)
        normalised_score = candidate.get("normalised_score", 0.0)
        # print(f"Candidate sentence: {candidate_sentence}")
        # Create EstNLTK Text object and perform morphological analysis
        estnltk_text = estnltk.Text(candidate_sentence)
        estnltk_text.tag_layer("morph_analysis")
        # Extract the morphological tags for the candidate word
        morph_layer = estnltk_text.morph_analysis
        # Find the token in the morph layer that corresponds to the candidate replacement
        candidate_token = None
        for token in morph_layer:
            if (
                token.start <= candidate_span[0] < token.end
            ):  # Check if the token covers the start of the original word span
                candidate_token = token
                break
        if candidate_token:
            # Count up the form labels for the candidate
            number_of_labels = len(candidate_token.form) if candidate_token.form else 0
            if number_of_labels > 0:
                # Distribute the normalised score equally among all form labels for this candidate
                weight_per_label = normalised_score / number_of_labels
                for candidate_label in candidate_token.form:
                    form_probabilities[candidate_label] = (
                        form_probabilities.get(candidate_label, 0) + weight_per_label
                    )
            else:
                form_probabilities["unknown"] = (
                    form_probabilities.get("unknown", 0) + normalised_score
                )

    best_form_label = (
        max(form_probabilities, key=form_probabilities.get)
        if form_probabilities
        else "unknown"
    )
    # Create a record for the chosen candidate and its features
    candidate_record = {
        "id": id,
        "candidates": candidates,
        "pred_label": best_form_label,
        "true_label": label,
        "predicted_form_distribution": form_probabilities,
        "source_sentence": source_sentence,
        "target_word": target_word,
        "word_span": word_span,
    }
    new_records.append(candidate_record)

100%|██████████| 7884/7884 [04:33<00:00, 28.79it/s]


In [15]:
# Create a new DataFrame from the predicted form labels and their distributions and save to JSON
homonym_results_df = pd.DataFrame(new_records)
homonym_results_df_path = (
    HOMONYMS_DIRS["processed"] / "homonyms_bert_mlm_llm_predictions_full.parquet"
)
homonym_results_df.to_parquet(homonym_results_df_path, index=False)

In [16]:
client.close()